# Single-chain model validation

In [ ]:
from pathlib import Path
from functools import partial
from dataclasses import replace
import sys
import pandas as pd
import matplotlib.pyplot as plt

here = Path.cwd().resolve()
ROOT = next(
    (p for p in (here, *here.parents)
     if (p / "analysis" / "src" / "plots" / "style.py").is_file()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter from inside the repository checkout.")
sys.path.insert(0, str(ROOT / "analysis"))

from src.plots.style import PlotStyle
from src.plots.data import load_sweep, filter_rows
from src.plots.panels import panels_by

STYLE = PlotStyle()
OUTPUT_ROOT = ROOT / "analysis_results"


In [ ]:
from src.plots.figures import (
    plot_mfe_pairing_scatter, plot_precision_recall, plot_rg_validation,
)

SWEEP = "53c_single_mol_duplex"
DENSITY = None
RG_REFERENCE_NM = 1.47

replicates = load_sweep(OUTPUT_ROOT, SWEEP, "replicates")
summary = load_sweep(OUTPUT_ROOT, SWEEP, "summary")
if DENSITY is not None:
    replicates = filter_rows(replicates, density=DENSITY)
    summary = filter_rows(summary, density=DENSITY)
E_VALUES = sorted(summary["E_att"].dropna().unique())

## MFE-matching versus additional base pairs

In [ ]:
fig, axes = panels_by(
    replicates, by="angle_k_ppp",
    plot_fn=partial(
        plot_mfe_pairing_scatter, style=STYLE, attraction_order=E_VALUES,
    ),
    style=STYLE, ncols=3, legend="last",
)
plt.show()

## Precision and recall

In [ ]:
fig, axes = panels_by(
    replicates, by="angle_k_ppp",
    plot_fn=partial(
        plot_precision_recall, style=STYLE, attraction_order=E_VALUES,
    ),
    style=STYLE, ncols=3, legend="last",
)
plt.show()

## Radius of gyration

In [ ]:
fig, axes = panels_by(
    summary, by="angle_k_ppp",
    plot_fn=lambda d, ax: plot_rg_validation(
        d, ax,
        replicates=filter_rows(
            replicates, angle_k_ppp=d["angle_k_ppp"].iloc[0],
        ),
        style=STYLE, x_order=E_VALUES, reference_nm=RG_REFERENCE_NM,
    ),
    style=STYLE, ncols=3, legend="last",
)
plt.show()